# Broad MC-Dropout Alignment Recovery

This notebook completes the broad `dropout_k8_p005` experiment after the scorer-indexing defect was identified.

It reuses the saved stochastic samples. The default path does **not** rerun model inference: it verifies that `score_index` is a complete non-identity permutation, reindexes only source-ordered reference arrays, regenerates strategies and reports, and records before/after alignment evidence.

## 0. Resource Assumptions

| Resource | Default requirement |
| --- | --- |
| Runtime | Colab CPU is sufficient; GPU is not used by the default repair |
| Input | Existing broad compact NPZ and manifest on Google Drive |
| Memory | About 4 GB RAM for 500K rows, K=8 |
| Disk | About 2 GB free locally and on Drive for corrected tables and bundle |
| Network | One pinned Git clone plus a small pinned Python overlay |
| Checkpoints | Not required for `repair_compact` |
| Optional fallback | `reaggregate_raw` reads existing raw score shards; it still does not run model inference |

The legacy artifact remains immutable. Corrected artifacts live under a separate `alignment_fixed` root.

## 1. Runtime and Google Drive

Mount Drive, inspect the runtime, and stop early if the project root is absent.

In [ ]:
# PYTHON CELL
from google.colab import drive
drive.mount('/content/drive')

import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/color-filter-ablation')
if not DRIVE_ROOT.is_dir():
    raise FileNotFoundError(f'Expected project root: {DRIVE_ROOT}')

print('python:', sys.version)
print('platform:', platform.platform())
print('cpu count:', os.cpu_count())
print('disk /content:', shutil.disk_usage('/content'))
print('disk drive:', shutil.disk_usage(DRIVE_ROOT))
subprocess.run(['nvidia-smi'], check=False)

## 2. Clone, Pin, and Install

The analysis code is checked out at an exact pushed commit. The historical producer revision was not recorded by the original notebook, so the provenance field says that explicitly rather than inventing a SHA.

In [ ]:
# PYTHON CELL
REPO_URL = 'https://github.com/myazdani/color-filter-olmo.git'
ANALYSIS_SHA = '846b800a6a0f0388371d5b586f1adcc7b5f91d37'
PRODUCER_REVISION = 'legacy-unrecorded-broad-dropout-k8-p005'
NOTEBOOK_REVISION = 'alignment-recovery-v1-2026-07-15'
OLMO_DIR = Path('/content/color-filter-olmo')

if not (OLMO_DIR / '.git').is_dir():
    subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, str(OLMO_DIR)], check=True)
subprocess.run(['git', '-C', str(OLMO_DIR), 'fetch', '--depth', '1', 'origin', 'uncertainty-color'], check=True)
subprocess.run(['git', '-C', str(OLMO_DIR), 'checkout', '--detach', ANALYSIS_SHA], check=True)
resolved = subprocess.check_output(['git', '-C', str(OLMO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
if resolved != ANALYSIS_SHA:
    raise RuntimeError((resolved, ANALYSIS_SHA))
print('analysis revision:', resolved)
print('producer revision:', PRODUCER_REVISION)
print('notebook revision:', NOTEBOOK_REVISION)

In [ ]:
# PYTHON CELL
PINNED_OVERLAY = [
    'pandas==2.2.2',
    'pyarrow==18.1.0',
    'matplotlib==3.10.0',
    'pytest==8.3.5',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *PINNED_OVERLAY], check=True)

import importlib
import importlib.metadata
sys.path.insert(0, str(OLMO_DIR))
recovery_module = importlib.import_module('scripts.dropout_uncertainty_recovery_colab')
recovery_module = importlib.reload(recovery_module)

for package in ('numpy', 'pandas', 'pyarrow', 'matplotlib', 'pytest'):
    print(package, importlib.metadata.version(package))
print('helper:', Path(recovery_module.__file__).resolve())

In [ ]:
# PYTHON CELL
metrics_script = OLMO_DIR / 'scripts/21_dropout_uncertainty_metrics.py'
strategy_script = OLMO_DIR / 'scripts/22_dropout_strategy_sweep.py'
helper_script = OLMO_DIR / 'scripts/dropout_uncertainty_recovery_colab.py'
for path in (metrics_script, strategy_script, helper_script):
    if not path.is_file():
        raise FileNotFoundError(path)

marker = 'metadata_and_full_scores_indexed_by_score_index'
if marker not in metrics_script.read_text(encoding='utf-8'):
    raise RuntimeError('Pinned aggregation code lacks the score_index alignment fix')
subprocess.run([sys.executable, str(strategy_script), '--help'], check=True, stdout=subprocess.PIPE)
print('capability probe passed:', marker)

## 3. Configure the Recovery

The default `repair_compact` mode is the intended path. Switch to `reaggregate_raw` only to reconstruct the compact artifact from existing score shards. Neither mode launches OLMo scoring.

In [ ]:
# PYTHON CELL
from scripts.dropout_uncertainty_recovery_colab import (
    DropoutUncertaintyRecovery,
    RecoveryContext,
)

CONFIG_ID = 'dropout_k8_p005'
RECOVERY_MODE = 'repair_compact'  # Optional fallback: 'reaggregate_raw'
RUN_ROOT = DRIVE_ROOT / 'results' / 'dropout-uncertainty' / CONFIG_ID
LEGACY_ANALYSIS_DIR = RUN_ROOT / 'analysis'
FIXED_ROOT = RUN_ROOT / 'alignment_fixed'
CORRECTED_ANALYSIS_DIR = FIXED_ROOT / 'analysis'
REPORT_DIR = FIXED_ROOT / 'report'
RAW_SCORE_ROOT = DRIVE_ROOT / 'results' / 'dropout-uncertainty' / 'raw_score_shards' / CONFIG_ID
METADATA_PATH = DRIVE_ROOT / 'data' / 'score_pool_meta_official_500k.parquet'
FULL_SCORES_PATH = DRIVE_ROOT / 'results' / 'score-pool-robustness-official-500k' / 'scores_full.parquet'

context = RecoveryContext(
    olmo_dir=OLMO_DIR,
    legacy_analysis_dir=LEGACY_ANALYSIS_DIR,
    corrected_analysis_dir=CORRECTED_ANALYSIS_DIR,
    report_dir=REPORT_DIR,
    raw_score_root=RAW_SCORE_ROOT,
    metadata_path=METADATA_PATH,
    full_scores_path=FULL_SCORES_PATH,
    producer_sha=PRODUCER_REVISION,
    analysis_sha=ANALYSIS_SHA,
    notebook_revision=NOTEBOOK_REVISION,
)
workflow = DropoutUncertaintyRecovery(context)
print('mode:', RECOVERY_MODE)
print('legacy input:', workflow.legacy_npz)
print('fixed root:', FIXED_ROOT)

## 4. Validate Inputs

This is a hard preflight. It rejects missing arrays, non-finite values, an invalid permutation, already-fixed artifacts, ambiguous legacy IDs, unbalanced pool metadata, and repairs that do not materially improve alignment.

In [ ]:
# PYTHON CELL
from pprint import pprint

preflight = workflow.preflight(RECOVERY_MODE)
pprint(preflight)
legacy_check = preflight['legacy']
if legacy_check['candidate_repaired_spearman_mean_vs_full'] <= legacy_check['raw_spearman_mean_vs_full'] + 0.20:
    raise RuntimeError('Alignment evidence is too weak for automatic repair')
print('legacy NPZ SHA-256:', legacy_check['legacy_npz_sha256'])

## 5. Model Conversion or Loading

Not applicable for the default recovery. The model outputs already exist and the stochastic sample tensors are reused byte-for-byte.

For the optional `reaggregate_raw` mode, this notebook reads prior/books raw score shards directly; checkpoints are still not loaded.

## 6. Cheap Synthetic Gate

Run the exact pushed regression that recreates a shuffled scorer permutation. It verifies source immutability, reference reindexing, strategy generation, report generation, selected-ID validity, and idempotent reuse.

In [ ]:
# PYTHON CELL
gate = OLMO_DIR / 'tests/dropout_uncertainty_recovery_colab_test.py'
subprocess.run(
    [
        sys.executable, '-m', 'pytest', '--confcutdir=tests', '-q',
        str(gate) + '::test_repair_preserves_samples_and_realigns_reference_arrays',
    ],
    cwd=OLMO_DIR,
    check=True,
)
print('synthetic alignment gate passed')

## 7. Batch and Shard Tuning

No GPU batch tuning is needed. The default repair streams compressed inputs through NumPy/Pandas. This bounded check confirms expected input size and available output capacity.

In [ ]:
# PYTHON CELL
legacy_bytes = workflow.legacy_npz.stat().st_size
free_local = shutil.disk_usage('/content').free
free_drive = shutil.disk_usage(DRIVE_ROOT).free
minimum_free = max(2_000_000_000, legacy_bytes * 8)
print('legacy bytes:', legacy_bytes)
print('minimum free bytes:', minimum_free)
print('local free:', free_local, 'drive free:', free_drive)
if free_local < minimum_free or free_drive < minimum_free:
    raise RuntimeError('Insufficient free space for corrected tables and bundle')

## 8. Full Resumable Recovery

Safe to rerun. The legacy source is never modified. A matching completed repair is reused; incomplete strategy outputs are regenerated in their isolated corrected directory.

Expected default runtime is dominated by 500K-row table serialization, strategy evaluation, and Drive I/O, not model inference.

In [ ]:
# PYTHON CELL
RUN_RECOVERY = True
if not RUN_RECOVERY:
    raise RuntimeError('Set RUN_RECOVERY=True after reviewing Sections 0-7')

run_result = workflow.run(mode=RECOVERY_MODE, write_parquet=True)
pprint(run_result)
print('corrected analysis:', CORRECTED_ANALYSIS_DIR)
print('corrected report:', REPORT_DIR)

## 9. Resume After Disconnect

After reconnecting, rerun Sections 1-4, then this status cell. If validation is incomplete, rerun Section 8; do not delete or overwrite the legacy analysis directory.

In [ ]:
# PYTHON CELL
status = workflow.status()
pprint(status)
if 'validation_error' in status:
    print('Recovery is incomplete. Rerun Section 8.')
else:
    print('Recovery is complete and validated; Section 8 may be rerun safely.')

## 10. Metrics and Report

Regenerate the report from corrected artifacts, display the main outputs, and enforce the task-level evidence gates.

In [ ]:
# PYTHON CELL
from IPython.display import Image, Markdown, display

report_result = workflow.build_report()
display(Markdown(Path(report_result['report']).read_text(encoding='utf-8')))
display(Image(filename=str(REPORT_DIR / 'alignment_corrected_results.png')))

In [ ]:
# PYTHON CELL
import json

summary = report_result['summary']
audit = json.loads(workflow.audit_path.read_text(encoding='utf-8'))
acceptance = {
    'corrected_spearman_gt_0_75': summary['spearman_mean_vs_full'] > 0.75,
    'alignment_lift_gt_0_20': (
        summary['spearman_mean_vs_full'] - audit['raw_spearman_mean_vs_full'] > 0.20
    ),
    'mean_pairwise_auc_gt_0_75': summary['mean_pairwise_auc'] > 0.75,
    'recall_at_1_64_gt_0_80': summary['recall_vs_full_1_64'] > 0.80,
    'stochastic_samples_unchanged': audit['stochastic_sample_arrays_changed'] is False,
}
pprint(acceptance)
if not all(acceptance.values()):
    raise RuntimeError(f'Corrected result failed acceptance gates: {acceptance}')

## 11. Output Review, Bundle, and Download

Validate the complete corrected artifact contract, then build a verified handoff archive. The archive excludes checkpoints, token arrays, raw scorer memmaps, and a duplicate copy of the legacy NPZ.

In [ ]:
# PYTHON CELL
validation = workflow.validate_corrected(require_strategy=True, require_parquet=True)
pprint(validation)

required_report_files = [
    REPORT_DIR / 'report.md',
    REPORT_DIR / 'report.html',
    REPORT_DIR / 'alignment_corrected_results.png',
    REPORT_DIR / 'report_manifest.json',
]
missing = [str(path) for path in required_report_files if not path.is_file() or path.stat().st_size == 0]
if missing:
    raise FileNotFoundError(missing)
print('output review passed')

In [ ]:
# PYTHON CELL
LOCAL_ARCHIVE = Path('/content/dropout_uncertainty_alignment_fixed_bundle.zip')
DRIVE_ARCHIVE = FIXED_ROOT / 'dropout_uncertainty_alignment_fixed_bundle.zip'
BUNDLE_INFO = workflow.build_bundle(LOCAL_ARCHIVE, DRIVE_ARCHIVE)
pprint({
    'archive_path': str(BUNDLE_INFO['archive_path']),
    'drive_archive_path': str(BUNDLE_INFO['drive_archive_path']),
    'file_count': BUNDLE_INFO['file_count'],
})

In [ ]:
# DOWNLOAD-ONLY CELL
AUTO_DOWNLOAD = True
if AUTO_DOWNLOAD:
    from google.colab import files
    files.download(str(BUNDLE_INFO['archive_path']))